# Amazon Products Sales Data Cleaning

This notebook handles the initial data cleaning for the Capstone 2 project. We will load the raw dataset, handle missing values, format data types, remove duplicates, and export the cleaned dataset for Exploratory Data Analysis (EDA) and Tableau visualization.

In [1]:
import pandas as pd
import numpy as np

/var/folders/8k/lcyh43612r54vw01bhc2hdkw0000gn/T/ipykernel_58456/2162656668.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1. Data Loading & Inspection

In [2]:
# Load the raw dataset
df = pd.read_csv('../data/raw/amazon_products_sales_data_raw.csv')

# Display basic information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_title         42675 non-null  object 
 1   product_rating        41651 non-null  float64
 2   total_reviews         41651 non-null  float64
 3   purchased_last_month  32164 non-null  float64
 4   discounted_price      40613 non-null  float64
 5   original_price        40613 non-null  float64
 6   is_best_seller        42675 non-null  object 
 7   is_sponsored          42675 non-null  object 
 8   has_coupon            42675 non-null  object 
 9   buy_box_availability  28022 non-null  object 
 10  delivery_date         30692 non-null  object 
 11  sustainability_tags   3408 non-null   object 
 12  product_image_url     42675 non-null  object 
 13  product_page_url      40606 non-null  object 
 14  data_collected_at     42675 non-null  object 
 15  product_category   

In [3]:
df.head()

,product_title,product_rating,total_reviews,purchased_last_month,discounted_price,original_price,is_best_seller,is_sponsored,has_coupon,buy_box_availability,delivery_date,sustainability_tags,product_image_url,product_page_url,data_collected_at,product_category,discount_percentage
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375.0,300.0,89.68,159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,2025-09-01,Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Phones,43.60
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457.0,6000.0,9.99,15.99,No Badge,Sponsored,No Coupon,Add to cart,2025-08-29,NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,37.52
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044.0,2000.0,314.00,349.00,No Badge,Sponsored,No Coupon,Add to cart,2025-09-01,NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,10.03
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,35882.0,10000.0,162.24,162.24,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,https://www.amazon.com/Apple-Cancellation-Tran...,2025-08-21 11:14:29,Phones,0.00
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,28988.0,10000.0,72.74,72.74,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,https://www.amazon.com/Apple-MX542LL-A-AirTag-...,2025-08-21 11:14:29,Phones,0.00


## 2. Handling Missing Values

- `purchased_last_month`: Missing values imply 0 sales, so we fill with 0.
- `sustainability_tags`: Missing values indicate no tags, fill with 'None'.
- Ratings & Reviews: We will inspect missing values and likely drop rows lacking ratings since this is a core metric.

In [4]:
# Check missing values
missing_before = df.isnull().sum()
print("Missing values before cleaning:")
print(missing_before[missing_before > 0])

Missing values before cleaning:
product_rating           1024
total_reviews            1024
purchased_last_month    10511
discounted_price         2062
original_price           2062
buy_box_availability    14653
delivery_date           11983
sustainability_tags     39267
product_page_url         2069
discount_percentage      2062
dtype: int64


In [5]:
# Fill missing purchased_last_month with '0'
df['purchased_last_month'] = df['purchased_last_month'].fillna('0')

# Fill missing sustainability_tags with 'None'
df['sustainability_tags'] = df['sustainability_tags'].fillna('None')

# For core metrics like rating, reviews, and price, if they are missing we drop them
# as imputing might heavily bias product-level analytics
df = df.dropna(subset=['product_rating', 'total_reviews', 'discounted_price', 'original_price'])

/var/folders/8k/lcyh43612r54vw01bhc2hdkw0000gn/T/ipykernel_58456/661593678.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['purchased_last_month'] = df['purchased_last_month'].fillna('0')
/var/folders/8k/lcyh43612r54vw01bhc2hdkw0000gn

## 3. Data Type Formatting

We need to ensure boolean columns are True/False, dates are datetime objects, and strings with numeric meaning (like '10K+') are converted to integers.

In [6]:
# Convert string numbers (e.g. '10k+', '500+') to integers for 'purchased_last_month'
def parse_purchased(val):
    if pd.isna(val):
        return 0
    val = str(val).lower().replace('+', '').replace(',', '')
    if 'k' in val:
        return int(float(val.replace('k', '')) * 1000)
    try:
        return int(val)
    except:
        return 0

df['purchased_last_month'] = df['purchased_last_month'].apply(parse_purchased)

# Format Boolean Columns
df['is_best_seller'] = df['is_best_seller'].apply(lambda x: True if pd.notna(x) and x != 'No Badge' else False)
df['is_sponsored'] = df['is_sponsored'].apply(lambda x: True if pd.notna(x) and x == 'Sponsored' else False)
df['has_coupon'] = df['has_coupon'].apply(lambda x: False if pd.isna(x) or x == 'No Coupon' else True)

# Convert Dates
df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')
df['data_collected_at'] = pd.to_datetime(df['data_collected_at'], errors='coerce')

/var/folders/8k/lcyh43612r54vw01bhc2hdkw0000gn/T/ipykernel_58456/4037274157.py:13: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['purchased_last_month'] = df['purchased_last_month'].apply(parse_purchased)
/var/folders/8k/lcyh43612r54vw01b

## 4. Handling Duplicates
Ensure no product is duplicated by dropping duplicates based on `product_page_url`.

In [7]:
duplicates_count = df.duplicated(subset=['product_page_url']).sum()
print(f"Found {duplicates_count} duplicate products. Removing them...")

df = df.drop_duplicates(subset=['product_page_url'])
print(f"Remaining rows: {len(df)}")

Found 2002 duplicate products. Removing them...
Remaining rows: 37588


## 5. Export Cleaned Dataset
Save to `data/processed/` for downstream use.

In [8]:
# Verify missing values again
print(df.isnull().sum())

# Save
output_path = '../data/processed/amazon_products_sales_data_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\nSuccessfully saved cleaned dataset to {output_path}")

product_title               0
product_rating              0
total_reviews               0
purchased_last_month        0
discounted_price            0
original_price              0
is_best_seller              0
is_sponsored                0
has_coupon                  0
buy_box_availability    12043
delivery_date            9372
sustainability_tags         0
product_image_url           0
product_page_url            1
data_collected_at           0
product_category            0
discount_percentage         0
dtype: int64



Successfully saved cleaned dataset to ../data/processed/amazon_products_sales_data_cleaned.csv
